In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


XGBoost bootstrap

In [ ]:
# ============================================================
# XGBoost best-valid-setting bootstrap
# No saved model weights needed
# Re-train best valid setting from CSV, then bootstrap test set
# ============================================================

import os
import json
import warnings
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    precision_recall_curve,
    auc,
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
except Exception:
    !pip install -q xgboost
    from xgboost import XGBClassifier


# ============================================================
# 1. Paths
# ============================================================

PROJECT_ROOT = "<path of project root>"

TRAIN_CSV = "<path of training set>"
VALID_CSV = "<path of valid set>"
TEST_CSV  = "<path of test set>"

# setting CSV
TOP15_CSV = "<path of best setting csv>"

OUTPUT_DIR = "output path>"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ID_COL = "CaseNo"
LABEL_COL = "EWAS_label"

N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42
STRATIFIED = True


BEST_SETTING_METRIC = "valid_f1"


THRESHOLD_MODE = "valid_f1"   # "valid_f1", "valid_j", or "0.5"


# ============================================================
# 2. Utility functions
# ============================================================

def parse_comma_list(value: Optional[str]) -> Optional[List[str]]:
    if value is None or str(value).strip() == "":
        return None
    return [x.strip() for x in str(value).split(",") if x.strip()]


def safe_auc(y_true, y_prob):
    try:
        y_true = np.asarray(y_true).astype(int)
        y_prob = np.asarray(y_prob).astype(float)

        if len(np.unique(y_true)) < 2:
            return np.nan

        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return np.nan


def safe_auprc(y_true, y_prob):
    try:
        precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
        return float(auc(recall_vals, precision_vals))
    except Exception:
        return np.nan


def calc_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {}
    metrics["threshold"] = float(threshold)
    metrics["auroc"] = safe_auc(y_true, y_prob)
    metrics["auprc"] = safe_auprc(y_true, y_prob)
    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    try:
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average=None,
            zero_division=0,
            labels=[0, 1],
        )
        metrics["precision"] = float(prec[1])
        metrics["recall"] = float(rec[1])
        metrics["specificity"] = float(rec[0])
        metrics["f1"] = float(f1[1])
    except Exception:
        metrics["precision"] = 0.0
        metrics["recall"] = 0.0
        metrics["specificity"] = 0.0
        metrics["f1"] = 0.0

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    except Exception:
        tn, fp, fn, tp = 0, 0, 0, 0

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)
    metrics["num_negative"] = int((y_true == 0).sum())
    metrics["num_positive"] = int((y_true == 1).sum())
    metrics["positive_rate"] = float((y_true == 1).mean())

    return metrics


def find_thresholds(y_true, y_prob, grid_size=1001):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    best_f1 = -1.0
    best_f1_thresh = 0.5
    best_j = -1.0
    best_j_thresh = 0.5

    for t in np.linspace(0.0, 1.0, grid_size):
        y_pred = (y_prob >= t).astype(int)

        try:
            prec, rec, f1, _ = precision_recall_fscore_support(
                y_true,
                y_pred,
                average=None,
                zero_division=0,
                labels=[0, 1],
            )

            pos_f1 = float(f1[1])
            sensitivity = float(rec[1])
            specificity = float(rec[0])
            j_score = sensitivity + specificity - 1.0
        except Exception:
            pos_f1 = 0.0
            j_score = -1.0

        if pos_f1 > best_f1:
            best_f1 = pos_f1
            best_f1_thresh = float(t)

        if j_score > best_j:
            best_j = j_score
            best_j_thresh = float(t)

    return {
        "valid_f1_thresh": float(best_f1_thresh),
        "valid_f1_at_thresh": float(best_f1),
        "valid_j_thresh": float(best_j_thresh),
        "valid_j_score": float(best_j),
    }


def compute_scale_pos_weight(y_train, strategy):
    y_train = np.asarray(y_train).astype(int)

    n_pos = max(1, int((y_train == 1).sum()))
    n_neg = max(1, int((y_train == 0).sum()))

    ratio = n_neg / n_pos

    strategy = str(strategy).lower()

    if strategy in ["none", "1", "false"]:
        return 1.0

    if strategy in ["auto", "ratio"]:
        return float(ratio)

    if strategy == "sqrt":
        return float(np.sqrt(ratio))

    return float(strategy)


# ============================================================
# 3. Feature builder
# ============================================================

class XGBFeatureBuilder:
    def __init__(
        self,
        id_col,
        label_col,
        cpg_cols,
        region_cols,
        cont_cols,
        cat_cols,
    ):
        self.id_col = id_col
        self.label_col = label_col
        self.cpg_cols = list(cpg_cols)
        self.region_cols = list(region_cols)
        self.cont_cols = list(cont_cols)
        self.cat_cols = list(cat_cols)

        self.numeric_imputers = {}
        self.cat_encoder = None
        self.cat_feature_names = []

    def fit(self, df):
        for group_name, cols in [
            ("methylation", self.cpg_cols),
            ("region", self.region_cols),
            ("continuous", self.cont_cols),
        ]:
            if len(cols) == 0:
                self.numeric_imputers[group_name] = None
                continue

            X = df[cols].apply(pd.to_numeric, errors="coerce")
            imputer = SimpleImputer(strategy="median")
            imputer.fit(X)
            self.numeric_imputers[group_name] = imputer

        if len(self.cat_cols) > 0:
            X_cat = df[self.cat_cols].fillna("__MISSING__").astype(str)

            try:
                encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
            except TypeError:
                encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

            encoder.fit(X_cat)
            self.cat_encoder = encoder
            self.cat_feature_names = list(encoder.get_feature_names_out(self.cat_cols))
        else:
            self.cat_encoder = None
            self.cat_feature_names = []

    def _transform_numeric(self, df, cols, group_name):
        if len(cols) == 0:
            return np.zeros((len(df), 0), dtype=np.float32), []

        X = df[cols].apply(pd.to_numeric, errors="coerce")
        imputer = self.numeric_imputers[group_name]
        X_imp = imputer.transform(X).astype(np.float32)

        return X_imp, list(cols)

    def _transform_cat(self, df):
        if len(self.cat_cols) == 0 or self.cat_encoder is None:
            return np.zeros((len(df), 0), dtype=np.float32), []

        X_cat = df[self.cat_cols].fillna("__MISSING__").astype(str)
        X_oh = self.cat_encoder.transform(X_cat).astype(np.float32)

        return X_oh, self.cat_feature_names

    def transform_group(self, df, feature_set):
        feature_set = str(feature_set).lower()

        X_methy, names_methy = self._transform_numeric(df, self.cpg_cols, "methylation")
        X_region, names_region = self._transform_numeric(df, self.region_cols, "region")
        X_cont, names_cont = self._transform_numeric(df, self.cont_cols, "continuous")
        X_cat, names_cat = self._transform_cat(df)

        X_clin = np.concatenate([X_cont, X_cat], axis=1)
        names_clin = names_cont + names_cat

        if feature_set == "clinical_only":
            return X_clin, names_clin

        if feature_set == "methylation_only":
            return X_methy, names_methy

        if feature_set == "methylation_region":
            return np.concatenate([X_methy, X_region], axis=1), names_methy + names_region

        if feature_set == "clinical_methylation":
            return np.concatenate([X_clin, X_methy], axis=1), names_clin + names_methy

        if feature_set == "all":
            return np.concatenate([X_clin, X_methy, X_region], axis=1), names_clin + names_methy + names_region

        raise ValueError(f"Unknown feature_set: {feature_set}")


# ============================================================
# 4. Resolve columns
# ============================================================

def resolve_columns(train_df):
    columns = list(train_df.columns)

    default_cat = ["SEX", "smoking", "DRK", "betel", "SPORT", "HTN_FAM", "edu"]
    cat_cols = [c for c in default_cat if c in train_df.columns]

    default_region = [f"ISLAND_{i}" for i in range(1, 6)]
    region_cols = [c for c in default_region if c in train_df.columns]

    cpg_cols = [c for c in columns if str(c).startswith("cg")]

    exclude = set([ID_COL, LABEL_COL] + cpg_cols + region_cols + cat_cols)
    cont_cols = [
        c for c in columns
        if c not in exclude and pd.api.types.is_numeric_dtype(train_df[c])
    ]

    return {
        "cpg_cols": cpg_cols,
        "region_cols": region_cols,
        "cont_cols": cont_cols,
        "cat_cols": cat_cols,
    }


# ============================================================
# 5. Load data and best valid setting
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)
test_df = pd.read_csv(TEST_CSV)

top_df = pd.read_csv(TOP15_CSV)

# Remove failed rows if column exists
if "failed" in top_df.columns:
    top_df = top_df[top_df["failed"] == False].copy()

# Best valid setting
top_df = top_df.sort_values(BEST_SETTING_METRIC, ascending=False).reset_index(drop=True)
best = top_df.iloc[0].to_dict()

print("Best valid setting:")
for k in [
    "trial_id",
    "trial_name",
    "feature_set",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "reg_lambda",
    "reg_alpha",
    "scale_pos_weight",
    "valid_threshold",
    "valid_auroc",
    "valid_auprc",
    "valid_f1",
    "test_auroc",
    "test_auprc",
    "test_f1",
]:
    if k in best:
        print(f"{k}: {best[k]}")

feature_set = best["feature_set"]

col_info = resolve_columns(train_df)

with open(os.path.join(OUTPUT_DIR, "selected_best_valid_setting.json"), "w") as f:
    json.dump({k: str(v) for k, v in best.items()}, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "feature_columns.json"), "w") as f:
    json.dump(col_info, f, indent=2)


# ============================================================
# 6. Build features
# ============================================================

builder = XGBFeatureBuilder(
    id_col=ID_COL,
    label_col=LABEL_COL,
    cpg_cols=col_info["cpg_cols"],
    region_cols=col_info["region_cols"],
    cont_cols=col_info["cont_cols"],
    cat_cols=col_info["cat_cols"],
)

builder.fit(train_df)

X_train, feature_names = builder.transform_group(train_df, feature_set)
X_valid, _ = builder.transform_group(valid_df, feature_set)
X_test, _ = builder.transform_group(test_df, feature_set)

y_train = train_df[LABEL_COL].astype(int).values
y_valid = valid_df[LABEL_COL].astype(int).values
y_test = test_df[LABEL_COL].astype(int).values

test_ids = test_df[ID_COL].astype(str).values if ID_COL in test_df.columns else np.arange(len(test_df)).astype(str)

print("\nFeature shapes:")
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)

print("\nClass counts:")
print("train pos/neg:", int((y_train == 1).sum()), int((y_train == 0).sum()))
print("valid pos/neg:", int((y_valid == 1).sum()), int((y_valid == 0).sum()))
print("test  pos/neg:", int((y_test == 1).sum()), int((y_test == 0).sum()))


# ============================================================
# 7. Re-train XGBoost using best valid setting
# ============================================================

scale_pos_weight_used = compute_scale_pos_weight(
    y_train,
    best.get("scale_pos_weight", "sqrt"),
)

model = XGBClassifier(
    n_estimators=int(best["n_estimators"]),
    max_depth=int(best["max_depth"]),
    learning_rate=float(best["learning_rate"]),
    subsample=float(best["subsample"]),
    colsample_bytree=float(best["colsample_bytree"]),
    min_child_weight=float(best["min_child_weight"]),
    reg_lambda=float(best["reg_lambda"]),
    reg_alpha=float(best["reg_alpha"]),
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=float(scale_pos_weight_used),
    random_state=0,
    n_jobs=-1,
    tree_method="hist",
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False,
)

valid_prob = model.predict_proba(X_valid)[:, 1]
test_prob = model.predict_proba(X_test)[:, 1]

thresholds = find_thresholds(y_valid, valid_prob, grid_size=1001)

if THRESHOLD_MODE == "valid_f1":
    threshold = thresholds["valid_f1_thresh"]
elif THRESHOLD_MODE == "valid_j":
    threshold = thresholds["valid_j_thresh"]
elif THRESHOLD_MODE == "0.5":
    threshold = 0.5
else:
    raise ValueError("THRESHOLD_MODE must be 'valid_f1', 'valid_j', or '0.5'.")

print("\nValidation-selected thresholds:")
print(json.dumps(thresholds, indent=2))
print(f"\nUsing bootstrap threshold: {THRESHOLD_MODE} = {threshold:.4f}")


# ============================================================
# 8. Save predictions
# ============================================================

pred_df = pd.DataFrame({
    "CaseNo": test_ids,
    "true_label": y_test,
    "predicted_prob": test_prob,
    "predicted_class_0.5": (test_prob >= 0.5).astype(int),
    "predicted_class_valid_j": (test_prob >= thresholds["valid_j_thresh"]).astype(int),
    "predicted_class_valid_f1": (test_prob >= thresholds["valid_f1_thresh"]).astype(int),
})

pred_path = os.path.join(OUTPUT_DIR, f"predictions_{feature_set}_test_retrained_best_valid.csv")
pred_df.to_csv(pred_path, index=False)

with open(os.path.join(OUTPUT_DIR, f"thresholds_{feature_set}_retrained_best_valid.json"), "w") as f:
    json.dump(thresholds, f, indent=2)

point_metrics = calc_metrics(y_test, test_prob, threshold)

print("\nPoint estimate on test set:")
for k, v in point_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")


# ============================================================
# 9. Bootstrap on test predictions
# ============================================================

rng = np.random.default_rng(BOOTSTRAP_SEED)

indices = np.arange(len(y_test))
pos_idx = indices[y_test == 1]
neg_idx = indices[y_test == 0]

rows = []

for b in range(N_BOOTSTRAP):
    if STRATIFIED:
        sampled_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        sampled_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        sampled_idx = np.concatenate([sampled_pos, sampled_neg])
        rng.shuffle(sampled_idx)
    else:
        sampled_idx = rng.choice(indices, size=len(indices), replace=True)

    y_b = y_test[sampled_idx]
    p_b = test_prob[sampled_idx]

    m = calc_metrics(y_b, p_b, threshold)
    m["bootstrap_id"] = b
    rows.append(m)

boot_df = pd.DataFrame(rows)

boot_path = os.path.join(OUTPUT_DIR, f"bootstrap_{feature_set}_test_samples.csv")
boot_df.to_csv(boot_path, index=False)


# ============================================================
# 10. Bootstrap CI summary
# ============================================================

metric_cols = [
    "auroc",
    "auprc",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "positive_rate",
]

summary_rows = []

for m in metric_cols:
    values = boot_df[m].replace([np.inf, -np.inf], np.nan).dropna().values

    summary_rows.append({
        "metric": m,
        "point": point_metrics[m],
        "boot_mean": float(np.mean(values)),
        "boot_std": float(np.std(values, ddof=1)),
        "ci_lower_2.5": float(np.percentile(values, 2.5)),
        "ci_upper_97.5": float(np.percentile(values, 97.5)),
        "n_bootstrap_valid": int(len(values)),
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(OUTPUT_DIR, f"bootstrap_{feature_set}_test_summary.csv")
summary_df.to_csv(summary_path, index=False)

summary_json = {
    "feature_set": feature_set,
    "selected_by": BEST_SETTING_METRIC,
    "threshold_mode": THRESHOLD_MODE,
    "threshold": float(threshold),
    "n_bootstrap": N_BOOTSTRAP,
    "stratified": STRATIFIED,
    "scale_pos_weight_used": float(scale_pos_weight_used),
    "selected_setting": {k: str(v) for k, v in best.items()},
    "point_metrics": point_metrics,
    "summary": summary_df.to_dict(orient="records"),
}

summary_json_path = os.path.join(OUTPUT_DIR, f"bootstrap_{feature_set}_test_summary.json")

with open(summary_json_path, "w") as f:
    json.dump(summary_json, f, indent=2)

print("\nSaved:")
print(pred_path)
print(boot_path)
print(summary_path)
print(summary_json_path)

print("\nBootstrap 95% CI summary:")
display(summary_df)

pure Transformer boostrap

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    precision_recall_curve,
    confusion_matrix,
    auc,
)

# ============================================================
# 1. Path
# ============================================================

CHECKPOINT_DIR = "<path of chcekpoint>"

PRED_CSV = os.path.join(CHECKPOINT_DIR, "predictions_all_test.csv")
BEST_THRESH_JSON = os.path.join(CHECKPOINT_DIR, "best_thresholds.json")

OUT_BOOT_CSV = os.path.join(CHECKPOINT_DIR, "bootstrap_test_samples.csv")
OUT_SUMMARY_CSV = os.path.join(CHECKPOINT_DIR, "bootstrap_test_summary.csv")
OUT_SUMMARY_JSON = os.path.join(CHECKPOINT_DIR, "bootstrap_test_summary.json")

N_BOOTSTRAP = 1000
SEED = 42

THRESHOLD_KEY = "valid_f1_thresh"   # or "valid_j_thresh"

STRATIFIED = True


# ============================================================
# 2. Metric functions
# ============================================================

def safe_auroc(y_true, y_prob):
    try:
        y_true = np.asarray(y_true).astype(int)
        y_prob = np.asarray(y_prob).astype(float)

        if len(np.unique(y_true)) < 2:
            return np.nan

        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return np.nan


def safe_auprc(y_true, y_prob):
    try:
        precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
        return float(auc(recall_vals, precision_vals))
    except Exception:
        return np.nan


def calc_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    out = {}
    out["threshold"] = float(threshold)
    out["auroc"] = safe_auroc(y_true, y_prob)
    out["auprc"] = safe_auprc(y_true, y_prob)
    out["accuracy"] = float(accuracy_score(y_true, y_pred))
    out["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average=None,
        zero_division=0,
        labels=[0, 1],
    )

    out["precision"] = float(prec[1])
    out["recall"] = float(rec[1])
    out["specificity"] = float(rec[0])
    out["f1"] = float(f1[1])

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    except Exception:
        tn, fp, fn, tp = 0, 0, 0, 0

    out["tn"] = int(tn)
    out["fp"] = int(fp)
    out["fn"] = int(fn)
    out["tp"] = int(tp)
    out["num_negative"] = int((y_true == 0).sum())
    out["num_positive"] = int((y_true == 1).sum())
    out["positive_rate"] = float((y_true == 1).mean())

    return out


# ============================================================
# 3. Load predictions and threshold
# ============================================================

pred_df = pd.read_csv(PRED_CSV)

if "true_label" not in pred_df.columns:
    raise ValueError("predictions CSV must contain column: true_label")

if "predicted_prob" not in pred_df.columns:
    raise ValueError("predictions CSV must contain column: predicted_prob")

y_true = pred_df["true_label"].values.astype(int)
y_prob = pred_df["predicted_prob"].values.astype(float)

with open(BEST_THRESH_JSON, "r") as f:
    best_info = json.load(f)

if THRESHOLD_KEY not in best_info:
    raise ValueError(f"{THRESHOLD_KEY} not found in {BEST_THRESH_JSON}")

threshold = float(best_info[THRESHOLD_KEY])

print("Loaded predictions:", PRED_CSV)
print("N =", len(y_true))
print("Positive =", int((y_true == 1).sum()))
print("Negative =", int((y_true == 0).sum()))
print(f"Using threshold: {THRESHOLD_KEY} = {threshold:.4f}")


# ============================================================
# 4. Point estimate on original test set
# ============================================================

point_metrics = calc_metrics(y_true, y_prob, threshold)

print("\nPoint estimate on original test set")
for k, v in point_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")


# ============================================================
# 5. Bootstrap
# ============================================================

rng = np.random.default_rng(SEED)

indices = np.arange(len(y_true))
pos_idx = indices[y_true == 1]
neg_idx = indices[y_true == 0]

rows = []

for b in range(N_BOOTSTRAP):
    if STRATIFIED:
        sampled_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        sampled_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        sampled_idx = np.concatenate([sampled_pos, sampled_neg])
        rng.shuffle(sampled_idx)
    else:
        sampled_idx = rng.choice(indices, size=len(indices), replace=True)

    y_b = y_true[sampled_idx]
    p_b = y_prob[sampled_idx]

    metrics_b = calc_metrics(y_b, p_b, threshold)
    metrics_b["bootstrap_id"] = b
    rows.append(metrics_b)

boot_df = pd.DataFrame(rows)
boot_df.to_csv(OUT_BOOT_CSV, index=False)

print("\nSaved bootstrap samples to:")
print(OUT_BOOT_CSV)


# ============================================================
# 6. 95% CI summary
# ============================================================

metric_cols = [
    "auroc",
    "auprc",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "positive_rate",
]

summary_rows = []

for m in metric_cols:
    values = boot_df[m].replace([np.inf, -np.inf], np.nan).dropna().values

    summary_rows.append({
        "metric": m,
        "point": point_metrics[m],
        "boot_mean": float(np.mean(values)),
        "boot_std": float(np.std(values, ddof=1)),
        "ci_lower_2.5": float(np.percentile(values, 2.5)),
        "ci_upper_97.5": float(np.percentile(values, 97.5)),
        "n_bootstrap_valid": int(len(values)),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False)

summary_json = {
    "checkpoint_dir": CHECKPOINT_DIR,
    "prediction_csv": PRED_CSV,
    "threshold_key": THRESHOLD_KEY,
    "threshold": threshold,
    "n_bootstrap": N_BOOTSTRAP,
    "stratified": STRATIFIED,
    "point_metrics": point_metrics,
    "summary": summary_df.to_dict(orient="records"),
}

with open(OUT_SUMMARY_JSON, "w") as f:
    json.dump(summary_json, f, indent=2)

print("\nSaved bootstrap summary to:")
print(OUT_SUMMARY_CSV)
print(OUT_SUMMARY_JSON)

print("\nBootstrap 95% CI summary:")
display(summary_df)

# Baseline: FT-T and TabM

In [ ]:
!pip install pytorch_tabular #ft-t
!pip install tabm      #tabM
!pip install git+https://github.com/yandex-research/tabm.git  #tabM
!pip install rtdl_num_embeddings  #tabM
!pip install catboost   #catboost

dataset prepare

In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

def prepare_multi_modal_data_3way(train_csv, valid_csv, test_csv, target_col='EWAS_label'):
    """
    Read the Train, Valid, Test CSV files and do a normalization.
    """
    print("Loading...")
    train_df = pd.read_csv(train_csv)
    valid_df = pd.read_csv(valid_csv)
    test_df = pd.read_csv(test_csv)

    # check column consistency
    train_cols = list(train_df.columns)
    if list(valid_df.columns) != train_cols or list(test_df.columns) != train_cols:
        raise ValueError("The cols of train/valid/test are inconsistent...")

    # classify column
    id_cols = ['CaseNo']
    methy_cols = [c for c in train_cols if c.startswith('cg')]
    metadata_cols = [c for c in train_cols if c.startswith('ISLAND_')]

    # clinical column
    clinical_cols = [c for c in train_cols if c not in (methy_cols + metadata_cols + id_cols + [target_col])]

    print(f"   - Methylation: {len(methy_cols)} features")
    print(f"   - Probe Metadata: {len(metadata_cols)} features")
    print(f"   - Clinical Others: {len(clinical_cols)} features")

    all_baseline_features = methy_cols + metadata_cols + clinical_cols

    # Standardization
    scaler = StandardScaler()
    train_df[all_baseline_features] = scaler.fit_transform(train_df[all_baseline_features])

    valid_df[all_baseline_features] = scaler.transform(valid_df[all_baseline_features])
    test_df[all_baseline_features] = scaler.transform(test_df[all_baseline_features])

    print(f"   - Train shape: {train_df.shape}")
    print(f"   - Valid shape: {valid_df.shape}")
    print(f"   - Test shape:  {test_df.shape}")

    return train_df, valid_df, test_df, methy_cols, metadata_cols, clinical_cols

In [ ]:
# Data Preparation
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_path = "<path of training set>"
valid_path = "<path of valid set>"
test_path = "<path of test set>"
train_df, valid_df, test_df, methy_cols, metadata_cols, clinical_cols = prepare_multi_modal_data_3way(train_csv=train_path,
                                                 valid_csv=valid_path,
                                                 test_csv=test_path)
target_col = 'EWAS_label'

all_baseline_features = methy_cols + metadata_cols + clinical_cols

train_labels = train_df[target_col].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

print(f"class weights [Class 0, Class 1]: {class_weights}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

catboost boostrap

In [ ]:
# ============================================================
# CatBoost best-valid-setting bootstrap
# No saved model weights needed
# Re-train best valid setting from CSV, then bootstrap test set
# ============================================================

import os
import json
import ast
import warnings
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. Path settings
# ============================================================

PROJECT_ROOT = "<path of project root>"

TOP15_CSV = "<path of best setting csv>"

OUTPUT_DIR = "<output path>"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 1000
STRATIFIED = True

# best valid setting selection
BEST_SETTING_SORT = ["valid_auroc", "valid_auprc", "valid_f1"]

# threshold mode:
# "csv"       = use threshold recorded in top15 CSV, usually 0.5 if your tuning used THRESHOLD_MODE="0.5"
# "valid_f1"  = recompute best F1 threshold on validation probabilities, then apply to test
# "0.5"       = fixed 0.5
THRESHOLD_MODE_FOR_BOOTSTRAP = "valid_f1"

# target / ID column
ID_COL = "CaseNo"


# ============================================================
# 2. Metric helpers
# ============================================================

def safe_auroc(y_true, y_prob):
    try:
        y_true = np.asarray(y_true).astype(int)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return np.nan


def safe_auprc(y_true, y_prob):
    try:
        return float(average_precision_score(y_true, y_prob))
    except Exception:
        return np.nan


def calc_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {}
    metrics["threshold"] = float(threshold)
    metrics["auroc"] = safe_auroc(y_true, y_prob)
    metrics["auprc"] = safe_auprc(y_true, y_prob)
    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average=None,
        zero_division=0,
        labels=[0, 1],
    )

    metrics["precision"] = float(prec[1])
    metrics["recall"] = float(rec[1])
    metrics["specificity"] = float(rec[0])
    metrics["f1"] = float(f1[1])

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    except Exception:
        tn, fp, fn, tp = 0, 0, 0, 0

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)

    metrics["num_negative"] = int((y_true == 0).sum())
    metrics["num_positive"] = int((y_true == 1).sum())
    metrics["positive_rate"] = float((y_true == 1).mean())

    return metrics


def find_best_f1_threshold(y_true, y_prob, grid_size=101):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    best_f1 = -1.0
    best_t = 0.5

    for t in np.linspace(0.0, 1.0, grid_size):
        y_pred = (y_prob >= t).astype(int)

        _, _, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average=None,
            zero_division=0,
            labels=[0, 1],
        )

        pos_f1 = float(f1[1])

        if pos_f1 > best_f1:
            best_f1 = pos_f1
            best_t = float(t)

    return best_t, best_f1


def parse_class_weights(value, y_train):
    """
    Reconstruct class_weights from CSV string.
    If parsing fails, compute balanced weights like your tuning code.
    """
    if value is not None and not pd.isna(value):
        try:
            parsed = ast.literal_eval(str(value))
            return {int(k): float(v) for k, v in parsed.items()}
        except Exception:
            pass

    labels = np.asarray(y_train).astype(int)
    n_neg = max(1, int((labels == 0).sum()))
    n_pos = max(1, int((labels == 1).sum()))
    total = n_neg + n_pos

    w0 = total / (2 * n_neg)
    w1 = total / (2 * n_pos)

    return {0: float(w0), 1: float(w1)}


# ============================================================
# 3. Load best valid setting
# ============================================================

top_df = pd.read_csv(TOP15_CSV)

if "failed" in top_df.columns:
    top_df = top_df[top_df["failed"] == False].copy()

for col in BEST_SETTING_SORT:
    if col in top_df.columns:
        top_df[col] = pd.to_numeric(top_df[col], errors="coerce")

top_df = top_df.sort_values(
    [c for c in BEST_SETTING_SORT if c in top_df.columns],
    ascending=False,
).reset_index(drop=True)

best = top_df.iloc[0].to_dict()

print("Best valid CatBoost setting:")
for k in [
    "trial_id",
    "trial_name",
    "valid_auroc",
    "valid_auprc",
    "valid_f1",
    "test_auroc",
    "test_auprc",
    "test_f1",
    "iterations",
    "learning_rate",
    "depth",
    "l2_leaf_reg",
    "random_strength",
    "border_count",
    "best_iteration",
    "valid_f1_threshold",
    "class_weights",
]:
    if k in best:
        print(f"{k}: {best[k]}")

with open(os.path.join(OUTPUT_DIR, "selected_best_valid_setting.json"), "w") as f:
    json.dump({k: str(v) for k, v in best.items()}, f, indent=2)


# ============================================================
# 4. Prepare data
# ============================================================

X_train = train_df[all_baseline_features].copy()
y_train = train_df[target_col].copy().astype(int).values

X_valid = valid_df[all_baseline_features].copy()
y_valid = valid_df[target_col].copy().astype(int).values

X_test = test_df[all_baseline_features].copy()
y_test = test_df[target_col].copy().astype(int).values

test_ids = (
    test_df[ID_COL].astype(str).values
    if ID_COL in test_df.columns
    else np.array([str(i) for i in range(len(test_df))])
)

cat_features = [
    col for col in all_baseline_features
    if str(X_train[col].dtype) in ["object", "category", "bool"]
]

for col in cat_features:
    X_train[col] = X_train[col].astype(str).fillna("__MISSING__")
    X_valid[col] = X_valid[col].astype(str).fillna("__MISSING__")
    X_test[col] = X_test[col].astype(str).fillna("__MISSING__")

print("\nData shapes:")
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test :", X_test.shape)
print("Categorical features:", cat_features)

print("\nClass distribution:")
print("train negative/positive:", int((y_train == 0).sum()), int((y_train == 1).sum()))
print("valid negative/positive:", int((y_valid == 0).sum()), int((y_valid == 1).sum()))
print("test  negative/positive:", int((y_test == 0).sum()), int((y_test == 1).sum()))


# ============================================================
# 5. Re-train CatBoost using best valid setting
# ============================================================

class_weights_dict = parse_class_weights(best.get("class_weights", None), y_train)

catboost_params = {
    "iterations": int(best["iterations"]),
    "learning_rate": float(best["learning_rate"]),
    "depth": int(best["depth"]),
    "l2_leaf_reg": float(best["l2_leaf_reg"]),
    "random_strength": float(best["random_strength"]),
    "border_count": int(best["border_count"]),
}

print("\nRe-training CatBoost with params:")
print(catboost_params)
print("class_weights:", class_weights_dict)

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="F1",
    class_weights=class_weights_dict,
    random_seed=RANDOM_SEED,
    verbose=False,
    allow_writing_files=False,
    **catboost_params,
)

fit_kwargs = {
    "X": X_train,
    "y": y_train,
    "eval_set": (X_valid, y_valid),
    "early_stopping_rounds": 50,
    "use_best_model": True,
}

if len(cat_features) > 0:
    fit_kwargs["cat_features"] = cat_features

model.fit(**fit_kwargs)

valid_probs = model.predict_proba(X_valid)[:, 1]
test_probs = model.predict_proba(X_test)[:, 1]

valid_f1_thresh, valid_best_f1 = find_best_f1_threshold(
    y_valid,
    valid_probs,
    grid_size=101,
)

if THRESHOLD_MODE_FOR_BOOTSTRAP == "valid_f1":
    threshold = valid_f1_thresh
elif THRESHOLD_MODE_FOR_BOOTSTRAP == "csv":
    threshold = float(best.get("valid_threshold", 0.5))
elif THRESHOLD_MODE_FOR_BOOTSTRAP == "0.5":
    threshold = 0.5
else:
    raise ValueError("THRESHOLD_MODE_FOR_BOOTSTRAP must be 'valid_f1', 'csv', or '0.5'")

print("\nValidation threshold:")
print("valid_f1_thresh:", valid_f1_thresh)
print("valid_best_f1_at_threshold:", valid_best_f1)
print("bootstrap threshold:", threshold)


# ============================================================
# 6. Save retrained predictions
# ============================================================

pred_df = pd.DataFrame({
    "CaseNo": test_ids,
    "true_label": y_test.astype(int),
    "predicted_prob": test_probs.astype(float),
    "predicted_class_0.5": (test_probs >= 0.5).astype(int),
    "predicted_class_valid_f1": (test_probs >= valid_f1_thresh).astype(int),
    "predicted_class_bootstrap_threshold": (test_probs >= threshold).astype(int),
})

pred_path = os.path.join(OUTPUT_DIR, "catboost_predictions_test_retrained_best_valid.csv")
pred_df.to_csv(pred_path, index=False)

threshold_info = {
    "threshold_mode_for_bootstrap": THRESHOLD_MODE_FOR_BOOTSTRAP,
    "threshold": float(threshold),
    "valid_f1_thresh": float(valid_f1_thresh),
    "valid_best_f1_at_threshold": float(valid_best_f1),
    "csv_valid_threshold": float(best.get("valid_threshold", 0.5)),
    "best_iteration_retrained": model.get_best_iteration(),
    "best_score_retrained": str(model.get_best_score()),
}

with open(os.path.join(OUTPUT_DIR, "catboost_thresholds_retrained_best_valid.json"), "w") as f:
    json.dump(threshold_info, f, indent=2)

point_metrics = calc_metrics(y_test, test_probs, threshold)

print("\nPoint estimate on test set:")
for k, v in point_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")


# ============================================================
# 7. Bootstrap test predictions
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

indices = np.arange(len(y_test))
pos_idx = indices[y_test == 1]
neg_idx = indices[y_test == 0]

rows = []

for b in range(N_BOOTSTRAP):
    if STRATIFIED:
        sampled_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        sampled_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        sampled_idx = np.concatenate([sampled_pos, sampled_neg])
        rng.shuffle(sampled_idx)
    else:
        sampled_idx = rng.choice(indices, size=len(indices), replace=True)

    y_b = y_test[sampled_idx]
    p_b = test_probs[sampled_idx]

    m = calc_metrics(y_b, p_b, threshold)
    m["bootstrap_id"] = b
    rows.append(m)

boot_df = pd.DataFrame(rows)

boot_path = os.path.join(OUTPUT_DIR, "catboost_bootstrap_test_samples.csv")
boot_df.to_csv(boot_path, index=False)


# ============================================================
# 8. Bootstrap 95% CI summary
# ============================================================

metric_cols = [
    "auroc",
    "auprc",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "positive_rate",
]

summary_rows = []

for m in metric_cols:
    values = boot_df[m].replace([np.inf, -np.inf], np.nan).dropna().values

    summary_rows.append({
        "metric": m,
        "point": point_metrics[m],
        "boot_mean": float(np.mean(values)),
        "boot_std": float(np.std(values, ddof=1)),
        "ci_lower_2.5": float(np.percentile(values, 2.5)),
        "ci_upper_97.5": float(np.percentile(values, 97.5)),
        "n_bootstrap_valid": int(len(values)),
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(OUTPUT_DIR, "catboost_bootstrap_test_summary.csv")
summary_df.to_csv(summary_path, index=False)

summary_json = {
    "selected_by": "valid_auroc",
    "threshold_mode_for_bootstrap": THRESHOLD_MODE_FOR_BOOTSTRAP,
    "threshold": float(threshold),
    "n_bootstrap": int(N_BOOTSTRAP),
    "stratified": bool(STRATIFIED),
    "selected_setting": {k: str(v) for k, v in best.items()},
    "retrained_catboost_params": catboost_params,
    "class_weights": class_weights_dict,
    "threshold_info": threshold_info,
    "point_metrics": point_metrics,
    "summary": summary_df.to_dict(orient="records"),
}

summary_json_path = os.path.join(OUTPUT_DIR, "catboost_bootstrap_test_summary.json")

with open(summary_json_path, "w") as f:
    json.dump(summary_json, f, indent=2)

print("\nSaved:")
print(pred_path)
print(boot_path)
print(summary_path)
print(summary_json_path)

print("\nBootstrap 95% CI summary:")
display(summary_df)

FT - Transformer boostrap

In [ ]:
# ============================================================
# FT-Transformer best-valid-setting bootstrap
# No saved weights needed
# Re-train best valid setting from CSV, then bootstrap test set
# ============================================================

import os
import json
import random
import shutil
import gc
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pytorch_tabular import TabularModel
from pytorch_tabular.config import DataConfig, TrainerConfig, OptimizerConfig

try:
    from pytorch_tabular.models import FTTransformerConfig
except Exception:
    from pytorch_tabular.models.ft_transformer.config import FTTransformerConfig

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. Paths / settings
# ============================================================

PROJECT_ROOT = "<path of project root>"


TRAIN_CSV = "<path of training set>"
VALID_CSV = "<path of valid set>"
TEST_CSV  = "path of test set>"


BEST_SETTING_CSV = "<path of best setting csv>"

OUTPUT_DIR = "<output path>"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
N_BOOTSTRAP = 1000
STRATIFIED = True

ID_COL = "CaseNo"
TARGET_COL_DEFAULT = "EWAS_label"


THRESHOLD_MODE_FOR_BOOTSTRAP = "valid_f1"  # "valid_f1", "csv", or "0.5"

FILL_CONTINUOUS_NA = True
CLEAN_TEMP_DIRS = True


# ============================================================
# 2. Utilities
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def clean_possible_temp_dirs():
    for d in ["saved_models", "lightning_logs", "wandb"]:
        if os.path.exists(d):
            try:
                shutil.rmtree(d)
            except Exception:
                pass


def safe_auroc(y_true, y_prob):
    try:
        y_true = np.asarray(y_true).astype(int)
        y_prob = np.asarray(y_prob).astype(float)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return np.nan


def safe_auprc(y_true, y_prob):
    try:
        y_true = np.asarray(y_true).astype(int)
        y_prob = np.asarray(y_prob).astype(float)
        return float(average_precision_score(y_true, y_prob))
    except Exception:
        return np.nan


def calc_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    y_pred = (y_prob >= threshold).astype(int)

    metrics = {}
    metrics["threshold"] = float(threshold)
    metrics["auroc"] = safe_auroc(y_true, y_prob)
    metrics["auprc"] = safe_auprc(y_true, y_prob)
    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average=None,
        zero_division=0,
        labels=[0, 1],
    )

    metrics["precision"] = float(prec[1])
    metrics["recall"] = float(rec[1])
    metrics["specificity"] = float(rec[0])
    metrics["f1"] = float(f1[1])

    try:
        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1],
        ).ravel()
    except Exception:
        tn, fp, fn, tp = 0, 0, 0, 0

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)
    metrics["num_negative"] = int((y_true == 0).sum())
    metrics["num_positive"] = int((y_true == 1).sum())
    metrics["positive_rate"] = float((y_true == 1).mean())

    return metrics


def find_best_f1_threshold(y_true, y_prob, grid_size=101):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    best_f1 = -1.0
    best_t = 0.5

    for t in np.linspace(0.0, 1.0, grid_size):
        y_pred = (y_prob >= t).astype(int)

        _, _, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average=None,
            zero_division=0,
            labels=[0, 1],
        )

        pos_f1 = float(f1[1])

        if pos_f1 > best_f1:
            best_f1 = pos_f1
            best_t = float(t)

    return best_t, best_f1


def extract_positive_probs(pred_df, target_col=None):

    cols = pred_df.columns.tolist()

    if target_col is not None:
        candidates = [
            f"{target_col}_1_probability",
            f"{target_col}_1_prob",
            f"{target_col}_probability_1",
            f"{target_col}_prob_1",
        ]

        for c in candidates:
            if c in pred_df.columns:
                return pred_df[c].values, c

    prob_cols = [
        col for col in cols
        if ("prob" in str(col).lower() or "probability" in str(col).lower())
    ]

    if len(prob_cols) == 0:
        raise ValueError(f"No probability columns found. Columns: {cols}")

    prob_col_1 = [
        col for col in prob_cols
        if str(col).endswith("_1_probability")
        or str(col).endswith("_1_prob")
        or "_1_" in str(col)
        or str(col).endswith("_1")
        or "class_1" in str(col).lower()
    ]

    if len(prob_col_1) > 0:
        prob_col = prob_col_1[0]
    elif len(prob_cols) >= 2:
        prob_col = prob_cols[-1]
    else:
        prob_col = prob_cols[0]

    return pred_df[prob_col].values, prob_col


def make_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): make_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [make_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    return obj


# ============================================================
# 3. Load data
# ============================================================

set_seed(RANDOM_SEED)

try:
    train_df
    valid_df
    test_df
    all_baseline_features
    target_col
    print("Using existing train_df / valid_df / test_df / all_baseline_features / target_col from notebook.")
except NameError:
    print("Loading train/valid/test CSV from paths...")
    train_df = pd.read_csv(TRAIN_CSV)
    valid_df = pd.read_csv(VALID_CSV)
    test_df = pd.read_csv(TEST_CSV)

    target_col = TARGET_COL_DEFAULT

    all_baseline_features = [
        c for c in train_df.columns
        if c not in [ID_COL, target_col]
    ]

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)
print("Target:", target_col)
print("Num features:", len(all_baseline_features))


# ============================================================
# 4. Load best valid setting
# ============================================================

best_df = pd.read_csv(BEST_SETTING_CSV)

if "failed" in best_df.columns:
    best_df = best_df[best_df["failed"] == False].copy()


for c in ["valid_f1", "valid_auprc", "valid_auroc", "valid_balanced_accuracy"]:
    if c in best_df.columns:
        best_df[c] = pd.to_numeric(best_df[c], errors="coerce")

best_df = best_df.sort_values(
    ["valid_f1", "valid_auprc", "valid_auroc", "valid_balanced_accuracy"],
    ascending=False,
).reset_index(drop=True)

best = best_df.iloc[0].to_dict()

print("\nSelected best valid FT-Transformer setting:")
for k in [
    "trial_id",
    "trial_name",
    "valid_f1",
    "valid_f1_threshold",
    "valid_auroc",
    "valid_auprc",
    "valid_balanced_accuracy",
    "test_f1",
    "test_auroc",
    "test_auprc",
    "input_embed_dim",
    "num_attn_blocks",
    "num_heads",
    "attn_dropout",
    "ff_dropout",
    "batch_size",
    "lr",
    "weight_decay",
    "max_epochs",
    "early_stopping_patience",
]:
    if k in best:
        print(f"{k}: {best[k]}")

with open(os.path.join(OUTPUT_DIR, "selected_best_valid_setting.json"), "w") as f:
    json.dump({k: str(v) for k, v in best.items()}, f, indent=2)


# ============================================================
# 5. Feature type / preprocessing
# ============================================================

train_df = train_df.copy()
valid_df = valid_df.copy()
test_df = test_df.copy()

for df_ in [train_df, valid_df, test_df]:
    df_[target_col] = df_[target_col].astype(int)

KNOWN_CAT_COLS = ["SEX", "smoking", "DRK", "betel", "SPORT", "HTN_FAM", "edu"]

auto_cat_cols = [
    c for c in all_baseline_features
    if str(train_df[c].dtype) in ["object", "category", "bool"]
]

categorical_cols = []

for c in KNOWN_CAT_COLS + auto_cat_cols:
    if c in all_baseline_features and c not in categorical_cols:
        categorical_cols.append(c)

continuous_cols = [
    c for c in all_baseline_features
    if c not in categorical_cols
]

print("\nCategorical cols:", categorical_cols)
print("Num categorical:", len(categorical_cols))
print("Num continuous :", len(continuous_cols))

for df_ in [train_df, valid_df, test_df]:
    for c in categorical_cols:
        df_[c] = df_[c].fillna("__MISSING__").astype(str)

if FILL_CONTINUOUS_NA:
    train_medians = {}

    for c in continuous_cols:
        train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
        valid_df[c] = pd.to_numeric(valid_df[c], errors="coerce")
        test_df[c] = pd.to_numeric(test_df[c], errors="coerce")

        med = train_df[c].median()
        if pd.isna(med):
            med = 0.0

        train_medians[c] = med

        train_df[c] = train_df[c].fillna(med)
        valid_df[c] = valid_df[c].fillna(med)
        test_df[c] = test_df[c].fillna(med)


y_train = train_df[target_col].values.astype(int)
y_valid = valid_df[target_col].values.astype(int)
y_test = test_df[target_col].values.astype(int)

test_ids = (
    test_df[ID_COL].astype(str).values
    if ID_COL in test_df.columns
    else np.array([str(i) for i in range(len(test_df))])
)

print("\nLabel distribution:")
print("train:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("valid:", pd.Series(y_valid).value_counts().sort_index().to_dict())
print("test :", pd.Series(y_test).value_counts().sort_index().to_dict())


# ============================================================
# 6. Weighted loss
# ============================================================

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
weighted_loss = nn.CrossEntropyLoss(weight=weights_tensor)

print("\nDevice:", device)
print("Class weights:", class_weights)


# ============================================================
# 7. Config builders
# ============================================================

def make_data_config():
    return DataConfig(
        target=[target_col],
        continuous_cols=continuous_cols,
        categorical_cols=categorical_cols,
    )


def make_model_config(cfg):
    model_kwargs = dict(
        task="classification",
        input_embed_dim=int(cfg["input_embed_dim"]),
        num_attn_blocks=int(cfg["num_attn_blocks"]),
        num_heads=int(cfg["num_heads"]),
        attn_dropout=float(cfg["attn_dropout"]),
        ff_dropout=float(cfg["ff_dropout"]),
        learning_rate=float(cfg["lr"]),
    )

    try:
        return FTTransformerConfig(
            **model_kwargs,
            ff_hidden_multiplier=4,
        )
    except TypeError:
        return FTTransformerConfig(**model_kwargs)


def make_optimizer_config(cfg):
    try:
        return OptimizerConfig(
            optimizer="AdamW",
            optimizer_params={
                "weight_decay": float(cfg["weight_decay"]),
            },
        )
    except TypeError:
        return OptimizerConfig(
            optimizer="AdamW",
        )


def make_trainer_config(cfg):
    common_kwargs = dict(
        batch_size=int(cfg["batch_size"]),
        max_epochs=int(cfg["max_epochs"]),
        early_stopping="valid_loss",
        early_stopping_mode="min",
        early_stopping_patience=int(cfg["early_stopping_patience"]),
        checkpoints="valid_loss",
        checkpoints_mode="min",
        load_best=True,
        progress_bar="none",
    )

    try:
        return TrainerConfig(
            accelerator="gpu" if torch.cuda.is_available() else "cpu",
            devices=1,
            **common_kwargs,
        )
    except TypeError:
        try:
            return TrainerConfig(
                gpus=1 if torch.cuda.is_available() else 0,
                **common_kwargs,
            )
        except TypeError:
            try:
                common_kwargs.pop("checkpoints_mode", None)
                return TrainerConfig(
                    gpus=1 if torch.cuda.is_available() else 0,
                    **common_kwargs,
                )
            except TypeError:
                common_kwargs.pop("checkpoints", None)
                common_kwargs.pop("checkpoints_mode", None)
                common_kwargs.pop("load_best", None)
                return TrainerConfig(**common_kwargs)


# ============================================================
# 8. Re-train best valid setting
# ============================================================

cfg = best.copy()

set_seed(RANDOM_SEED)

if CLEAN_TEMP_DIRS:
    clean_possible_temp_dirs()

data_config = make_data_config()
model_config = make_model_config(cfg)
optimizer_config = make_optimizer_config(cfg)
trainer_config = make_trainer_config(cfg)

tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

tabular_model.fit(
    train=train_df,
    validation=valid_df,
    loss=weighted_loss,
)

valid_pred_df = tabular_model.predict(valid_df)
test_pred_df = tabular_model.predict(test_df)

valid_probs, valid_prob_col = extract_positive_probs(
    valid_pred_df,
    target_col=target_col,
)

test_probs, test_prob_col = extract_positive_probs(
    test_pred_df,
    target_col=target_col,
)

valid_f1_thresh, valid_best_f1 = find_best_f1_threshold(
    y_valid,
    valid_probs,
    grid_size=101,
)

if THRESHOLD_MODE_FOR_BOOTSTRAP == "valid_f1":
    threshold = valid_f1_thresh
elif THRESHOLD_MODE_FOR_BOOTSTRAP == "csv":
    threshold = float(best.get("valid_f1_threshold", 0.5))
elif THRESHOLD_MODE_FOR_BOOTSTRAP == "0.5":
    threshold = 0.5
else:
    raise ValueError("THRESHOLD_MODE_FOR_BOOTSTRAP must be 'valid_f1', 'csv', or '0.5'.")

print("\nPrediction probability columns:")
print("valid:", valid_prob_col)
print("test :", test_prob_col)

print("\nValidation threshold:")
print("valid_f1_thresh:", valid_f1_thresh)
print("valid_best_f1_at_threshold:", valid_best_f1)
print("bootstrap threshold:", threshold)


# ============================================================
# 9. Save retrained predictions
# ============================================================

pred_df = pd.DataFrame({
    "CaseNo": test_ids,
    "true_label": y_test.astype(int),
    "predicted_prob": test_probs.astype(float),
    "predicted_class_0.5": (test_probs >= 0.5).astype(int),
    "predicted_class_valid_f1": (test_probs >= valid_f1_thresh).astype(int),
    "predicted_class_bootstrap_threshold": (test_probs >= threshold).astype(int),
})

pred_path = os.path.join(OUTPUT_DIR, "ft_transformer_predictions_test_retrained_best_valid.csv")
pred_df.to_csv(pred_path, index=False)

threshold_info = {
    "threshold_mode_for_bootstrap": THRESHOLD_MODE_FOR_BOOTSTRAP,
    "threshold": float(threshold),
    "valid_f1_thresh": float(valid_f1_thresh),
    "valid_best_f1_at_threshold": float(valid_best_f1),
    "csv_valid_f1_threshold": float(best.get("valid_f1_threshold", 0.5)),
    "valid_prob_col": valid_prob_col,
    "test_prob_col": test_prob_col,
}

with open(os.path.join(OUTPUT_DIR, "ft_transformer_thresholds_retrained_best_valid.json"), "w") as f:
    json.dump(threshold_info, f, indent=2)

point_metrics = calc_metrics(y_test, test_probs, threshold)

print("\nPoint estimate on test set:")
for k, v in point_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")


# ============================================================
# 10. Bootstrap test predictions
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

indices = np.arange(len(y_test))
pos_idx = indices[y_test == 1]
neg_idx = indices[y_test == 0]

rows = []

for b in range(N_BOOTSTRAP):
    if STRATIFIED:
        sampled_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        sampled_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        sampled_idx = np.concatenate([sampled_pos, sampled_neg])
        rng.shuffle(sampled_idx)
    else:
        sampled_idx = rng.choice(indices, size=len(indices), replace=True)

    y_b = y_test[sampled_idx]
    p_b = test_probs[sampled_idx]

    m = calc_metrics(y_b, p_b, threshold)
    m["bootstrap_id"] = b
    rows.append(m)

boot_df = pd.DataFrame(rows)

boot_path = os.path.join(OUTPUT_DIR, "ft_transformer_bootstrap_test_samples.csv")
boot_df.to_csv(boot_path, index=False)


# ============================================================
# 11. Bootstrap 95% CI summary
# ============================================================

metric_cols = [
    "auroc",
    "auprc",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "positive_rate",
]

summary_rows = []

for m in metric_cols:
    values = boot_df[m].replace([np.inf, -np.inf], np.nan).dropna().values

    summary_rows.append({
        "metric": m,
        "point": point_metrics[m],
        "boot_mean": float(np.mean(values)),
        "boot_std": float(np.std(values, ddof=1)),
        "ci_lower_2.5": float(np.percentile(values, 2.5)),
        "ci_upper_97.5": float(np.percentile(values, 97.5)),
        "n_bootstrap_valid": int(len(values)),
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(OUTPUT_DIR, "ft_transformer_bootstrap_test_summary.csv")
summary_df.to_csv(summary_path, index=False)

summary_json = {
    "selected_by": "valid_f1",
    "threshold_mode_for_bootstrap": THRESHOLD_MODE_FOR_BOOTSTRAP,
    "threshold": float(threshold),
    "n_bootstrap": int(N_BOOTSTRAP),
    "stratified": bool(STRATIFIED),
    "selected_setting": {k: str(v) for k, v in best.items()},
    "threshold_info": threshold_info,
    "point_metrics": point_metrics,
    "summary": summary_df.to_dict(orient="records"),
}

summary_json_path = os.path.join(OUTPUT_DIR, "ft_transformer_bootstrap_test_summary.json")

with open(summary_json_path, "w") as f:
    json.dump(summary_json, f, indent=2)

print("\nSaved:")
print(pred_path)
print(boot_path)
print(summary_path)
print(summary_json_path)

print("\nBootstrap 95% CI summary:")
display(summary_df)

# cleanup
try:
    del tabular_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
except Exception:
    pass

TabM boostrap

In [ ]:
# ============================================================
# TabM best-valid-setting bootstrap
# - Re-train best valid setting from CSV
# - Flexible threshold mode: 0.5 / valid_f1 / csv
# ============================================================

import os
import gc
import copy
import random
import json
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import tabm
from rtdl_num_embeddings import LinearReLUEmbeddings

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. Path / settings
# ============================================================

PROJECT_ROOT = "<path of project root>"

TOP15_CSV = "<path of best setting csv>"

OUTPUT_DIR = "<output path>"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_BOOTSTRAP = 1000
STRATIFIED = True

NUM_WORKERS = 0

TRAIN_CSV = "<path of training set>"
VALID_CSV = "<path of valid set>"
TEST_CSV  = "<path of test set>"

ID_COL = "CaseNo"
TARGET_COL = "EWAS_label"

THRESHOLD_MODE_FOR_BOOTSTRAP = "valid_f1"

BEST_SETTING_SORT = ["valid_auroc", "valid_auprc", "valid_f1"]

STANDARDIZE_FEATURES = False


# ============================================================
# 2. Utility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): make_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [make_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    return obj


def safe_get_str(row_dict, key, default):
    value = row_dict.get(key, default)
    if value is None or pd.isna(value):
        return default
    return str(value)


def safe_get_float(row_dict, key, default):
    value = row_dict.get(key, default)
    if value is None or pd.isna(value):
        return float(default)
    return float(value)


# ============================================================
# 3. Feature preprocessing
# Same logic as original TabM tuning
# ============================================================

def build_aligned_numeric_matrices(train_df, valid_df, test_df, feature_cols, standardize=False):
    train_X_raw = train_df[feature_cols].copy()
    valid_X_raw = valid_df[feature_cols].copy()
    test_X_raw  = test_df[feature_cols].copy()

    cat_cols = [
        c for c in feature_cols
        if str(train_X_raw[c].dtype) in ["object", "category", "bool"]
    ]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    print("Num numeric cols:", len(num_cols))
    print("Num categorical cols:", len(cat_cols))
    if len(cat_cols) > 0:
        print("Categorical cols:", cat_cols)

    # Numeric
    train_num = train_X_raw[num_cols].apply(pd.to_numeric, errors="coerce")
    valid_num = valid_X_raw[num_cols].apply(pd.to_numeric, errors="coerce")
    test_num  = test_X_raw[num_cols].apply(pd.to_numeric, errors="coerce")

    medians = train_num.median().fillna(0.0)

    train_num = train_num.fillna(medians)
    valid_num = valid_num.fillna(medians)
    test_num  = test_num.fillna(medians)

    # Categorical one-hot
    if len(cat_cols) > 0:
        train_cat = train_X_raw[cat_cols].astype(str).fillna("__MISSING__")
        valid_cat = valid_X_raw[cat_cols].astype(str).fillna("__MISSING__")
        test_cat  = test_X_raw[cat_cols].astype(str).fillna("__MISSING__")

        train_cat_ohe = pd.get_dummies(train_cat, dummy_na=False)
        valid_cat_ohe = pd.get_dummies(valid_cat, dummy_na=False)
        test_cat_ohe  = pd.get_dummies(test_cat, dummy_na=False)

        valid_cat_ohe = valid_cat_ohe.reindex(columns=train_cat_ohe.columns, fill_value=0)
        test_cat_ohe  = test_cat_ohe.reindex(columns=train_cat_ohe.columns, fill_value=0)

        train_X = pd.concat([train_num, train_cat_ohe], axis=1)
        valid_X = pd.concat([valid_num, valid_cat_ohe], axis=1)
        test_X  = pd.concat([test_num, test_cat_ohe], axis=1)
    else:
        train_X = train_num
        valid_X = valid_num
        test_X  = test_num

    # Standardize if needed
    if standardize:
        mean = train_X.mean(axis=0)
        std = train_X.std(axis=0).replace(0, 1.0).fillna(1.0)

        train_X = (train_X - mean) / std
        valid_X = (valid_X - mean) / std
        test_X  = (test_X - mean) / std

    train_np = train_X.astype(np.float32).values
    valid_np = valid_X.astype(np.float32).values
    test_np  = test_X.astype(np.float32).values

    feature_names = train_X.columns.tolist()

    return train_np, valid_np, test_np, feature_names


class EWASArrayDataset(Dataset):
    def __init__(self, X, y):
        self.x = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


# ============================================================
# 4. Metrics
# ============================================================

def safe_auroc(y_true, y_prob):
    try:
        y_true = np.asarray(y_true).astype(int)
        y_prob = np.asarray(y_prob).astype(float)
        if len(np.unique(y_true)) < 2:
            return np.nan
        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return np.nan


def safe_auprc(y_true, y_prob):
    try:
        y_true = np.asarray(y_true).astype(int)
        y_prob = np.asarray(y_prob).astype(float)
        return float(average_precision_score(y_true, y_prob))
    except Exception:
        return np.nan


def calc_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    y_pred = (y_prob >= threshold).astype(int)

    metrics = {}
    metrics["threshold"] = float(threshold)
    metrics["auroc"] = safe_auroc(y_true, y_prob)
    metrics["auprc"] = safe_auprc(y_true, y_prob)
    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average=None,
        zero_division=0,
        labels=[0, 1],
    )

    metrics["precision"] = float(prec[1])
    metrics["recall"] = float(rec[1])
    metrics["specificity"] = float(rec[0])
    metrics["f1"] = float(f1[1])

    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    except Exception:
        tn, fp, fn, tp = 0, 0, 0, 0

    metrics["tn"] = int(tn)
    metrics["fp"] = int(fp)
    metrics["fn"] = int(fn)
    metrics["tp"] = int(tp)
    metrics["num_negative"] = int((y_true == 0).sum())
    metrics["num_positive"] = int((y_true == 1).sum())
    metrics["positive_rate"] = float((y_true == 1).mean())

    return metrics


def find_best_f1_threshold(y_true, y_prob, grid_size=101):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    best_f1 = -1.0
    best_t = 0.5

    for t in np.linspace(0.0, 1.0, grid_size):
        y_pred = (y_prob >= t).astype(int)

        _, _, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average=None,
            zero_division=0,
            labels=[0, 1],
        )

        pos_f1 = float(f1[1])

        if pos_f1 > best_f1:
            best_f1 = pos_f1
            best_t = float(t)

    return best_t, best_f1


# ============================================================
# 5. TabM model helpers
# ============================================================

def make_loaders(batch_size):
    train_ds = EWASArrayDataset(X_train_np, y_train_np)
    valid_ds = EWASArrayDataset(X_valid_np, y_valid_np)
    test_ds  = EWASArrayDataset(X_test_np, y_test_np)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    valid_loader = DataLoader(
        valid_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, valid_loader, test_loader


def build_tabm_model(cfg):
    model = tabm.TabM.make(
        n_num_features=n_features,
        num_embeddings=LinearReLUEmbeddings(n_features),
        d_out=2,
        k=int(cfg["k"]),
    ).to(DEVICE)

    return model


def tabm_loss(logits, y, criterion):
    # logits shape: [batch_size, k, num_classes]
    k = logits.shape[1]
    loss = sum(criterion(logits[:, i, :], y) for i in range(k)) / k
    return loss


@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    all_probs = []
    all_targets = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x)
        loss = tabm_loss(logits, y, criterion)

        probs = F.softmax(logits, dim=-1)
        final_probs = probs.mean(dim=1)

        total_loss += loss.item() * x.size(0)
        all_probs.extend(final_probs[:, 1].detach().cpu().numpy().tolist())
        all_targets.extend(y.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(loader.dataset))
    all_probs = np.asarray(all_probs, dtype=float)
    all_targets = np.asarray(all_targets, dtype=int)

    return avg_loss, all_targets, all_probs


def choose_monitor(valid_loss, valid_labels, valid_probs, early_stopping_metric):
    if early_stopping_metric == "loss":
        return -float(valid_loss)
    if early_stopping_metric == "auroc":
        return safe_auroc(valid_labels, valid_probs)
    if early_stopping_metric == "f1":
        # Keep the same as your original tuning code: F1@0.5 for early stopping.
        metrics = calc_metrics(valid_labels, valid_probs, threshold=0.5)
        return float(metrics["f1"])
    raise ValueError(f"Unknown early_stopping_metric: {early_stopping_metric}")


def train_one_tabm_setting_for_bootstrap(cfg, early_stopping_metric):
    batch_size = int(cfg["batch_size"])

    train_loader, valid_loader, test_loader = make_loaders(batch_size)

    model = build_tabm_model(cfg)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg["lr"]),
        weight_decay=float(cfg["weight_decay"]),
    )

    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    max_epochs = int(cfg["max_epochs"])
    patience = int(cfg["patience"])

    best_monitor = -float("inf")
    best_epoch = -1
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            logits = model(x)
            loss = tabm_loss(logits, y, criterion)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x.size(0)

        train_loss /= max(1, len(train_loader.dataset))

        valid_loss, valid_labels, valid_probs = evaluate_loader(model, valid_loader, criterion)
        monitor_value = choose_monitor(
            valid_loss,
            valid_labels,
            valid_probs,
            early_stopping_metric=early_stopping_metric,
        )

        if monitor_value > best_monitor:
            best_monitor = monitor_value
            best_epoch = epoch
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        print(
            f"Epoch {epoch+1:02d}/{max_epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"valid_loss={valid_loss:.4f} | "
            f"monitor({early_stopping_metric})={monitor_value:.4f}"
        )

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_weights)

    valid_loss, valid_labels, valid_probs = evaluate_loader(model, valid_loader, criterion)
    test_loss, test_labels, test_probs = evaluate_loader(model, test_loader, criterion)

    return {
        "valid_loss": valid_loss,
        "test_loss": test_loss,
        "valid_labels": valid_labels,
        "valid_probs": valid_probs,
        "test_labels": test_labels,
        "test_probs": test_probs,
        "best_epoch": int(best_epoch + 1),
        "best_monitor": float(best_monitor),
    }


# ============================================================
# 6. Load data using your prepare_multi_modal_data_3way
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import pandas as pd
import torch
import os
import json

def prepare_multi_modal_data_3way(train_csv, valid_csv, test_csv, target_col="EWAS_label"):
    """
    Read Train / Valid / Test CSV files and apply StandardScaler.
    Scaler is fit on train only, then applied to valid/test.
    """
    print("Loading...")
    train_df = pd.read_csv(train_csv)
    valid_df = pd.read_csv(valid_csv)
    test_df = pd.read_csv(test_csv)

    # 1. Column consistency check
    train_cols = list(train_df.columns)
    if list(valid_df.columns) != train_cols or list(test_df.columns) != train_cols:
        raise ValueError("The cols of train/valid/test are inconsistent.")

    # 2. Feature grouping
    id_cols = ["CaseNo"]
    methy_cols = [c for c in train_cols if c.startswith("cg")]
    metadata_cols = [c for c in train_cols if c.startswith("ISLAND_")]

    clinical_cols = [
        c for c in train_cols
        if c not in (methy_cols + metadata_cols + id_cols + [target_col])
    ]

    print(f"   - Methylation: {len(methy_cols)} features")
    print(f"   - Probe Metadata / ISLAND: {len(metadata_cols)} features")
    print(f"   - Clinical Others: {len(clinical_cols)} features")

    all_baseline_features = methy_cols + metadata_cols + clinical_cols

    # 3. Make sure all features are numeric
    for df_name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
        for c in all_baseline_features:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # 4. Fill NA using train median before scaling
    train_medians = train_df[all_baseline_features].median().fillna(0.0)

    train_df[all_baseline_features] = train_df[all_baseline_features].fillna(train_medians)
    valid_df[all_baseline_features] = valid_df[all_baseline_features].fillna(train_medians)
    test_df[all_baseline_features] = test_df[all_baseline_features].fillna(train_medians)

    # 5. Standardization: fit train only
    scaler = StandardScaler()
    train_df[all_baseline_features] = scaler.fit_transform(train_df[all_baseline_features])
    valid_df[all_baseline_features] = scaler.transform(valid_df[all_baseline_features])
    test_df[all_baseline_features] = scaler.transform(test_df[all_baseline_features])

    print(f"   - Train shape: {train_df.shape}")
    print(f"   - Valid shape: {valid_df.shape}")
    print(f"   - Test shape:  {test_df.shape}")
    print(f"   - Final TabM input features: {len(all_baseline_features)}")

    return train_df, valid_df, test_df, methy_cols, metadata_cols, clinical_cols, all_baseline_features, scaler


# ============================================================
# Path settings
# ============================================================

train_path = TRAIN_CSV
valid_path = VALID_CSV
test_path  = TEST_CSV


target_col = "EWAS_label"

train_df, valid_df, test_df, methy_cols, metadata_cols, clinical_cols, all_baseline_features, scaler = prepare_multi_modal_data_3way(
    train_csv=train_path,
    valid_csv=valid_path,
    test_csv=test_path,
    target_col=target_col,
)

# ============================================================
# Load best valid TabM setting
# ============================================================

top_df = pd.read_csv(TOP15_CSV)

if "failed" in top_df.columns:
    top_df = top_df[top_df["failed"] == False].copy()

for c in BEST_SETTING_SORT:
    if c in top_df.columns:
        top_df[c] = pd.to_numeric(top_df[c], errors="coerce")

sort_cols = [c for c in BEST_SETTING_SORT if c in top_df.columns]

if len(sort_cols) == 0:
    raise ValueError(f"No sort columns found from BEST_SETTING_SORT={BEST_SETTING_SORT}")

top_df = top_df.sort_values(sort_cols, ascending=False).reset_index(drop=True)
best = top_df.iloc[0].to_dict()

print("\nSelected best valid TabM setting:")
for k in [
    "trial_id", "trial_name",
    "valid_auroc", "valid_auprc", "valid_f1",
    "test_auroc", "test_auprc", "test_f1",
    "k", "batch_size", "lr", "weight_decay", "max_epochs", "patience",
    "best_epoch", "best_monitor", "valid_f1_threshold", "valid_threshold",
    "early_stopping_metric", "n_features",
]:
    if k in best:
        print(f"{k}: {best[k]}")

with open(os.path.join(OUTPUT_DIR, "selected_best_valid_setting.json"), "w") as f:
    json.dump({k: str(v) for k, v in best.items()}, f, indent=2)


# ============================================================
# 7. Directly build TabM input from your prepared data
# ============================================================

X_train_np = train_df[all_baseline_features].astype(np.float32).values
X_valid_np = valid_df[all_baseline_features].astype(np.float32).values
X_test_np  = test_df[all_baseline_features].astype(np.float32).values

y_train_np = train_df[target_col].values.astype(int)
y_valid_np = valid_df[target_col].values.astype(int)
y_test_np  = test_df[target_col].values.astype(int)

n_features = X_train_np.shape[1]
final_feature_names = list(all_baseline_features)

test_ids = (
    test_df["CaseNo"].astype(str).values
    if "CaseNo" in test_df.columns
    else np.array([str(i) for i in range(len(test_df))])
)

print("\nFeature shapes:")
print("Train X:", X_train_np.shape)
print("Valid X:", X_valid_np.shape)
print("Test  X:", X_test_np.shape)
print("Final feature dim:", n_features)

print("\nLabel distribution:")
print("train:", pd.Series(y_train_np).value_counts().sort_index().to_dict())
print("valid:", pd.Series(y_valid_np).value_counts().sort_index().to_dict())
print("test :", pd.Series(y_test_np).value_counts().sort_index().to_dict())

# ============================================================
# Feature dimension check
# ============================================================

expected_n_features = best.get("n_features", None)

if expected_n_features is not None and not pd.isna(expected_n_features):
    expected_n_features = int(expected_n_features)

    if expected_n_features != n_features:
        print("\n[Warning] Feature dimension mismatch!")
        print(f"  CSV n_features      = {expected_n_features}")
        print(f"  Current n_features  = {n_features}")
        print("  This means the top15 CSV was probably produced with a different preprocessing pipeline.")
        print("  If your original TabM finetune used prepare_multi_modal_data_3way(), then the CSV should also have n_features =", n_features)
        print("  If the original CSV had n_features=242, it likely used one-hot categorical expansion.")
        print("  This run will continue using the current prepare_multi_modal_data_3way() pipeline.")

# ============================================================
# 8. Class weights
# ============================================================

classes = np.unique(y_train_np)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_np,
)

weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

print("\nClasses:", classes)
print("Class weights:", class_weights)


# ============================================================
# 9. Re-train best valid setting
# ============================================================

cfg = {
    "k": int(best["k"]),
    "batch_size": int(best["batch_size"]),
    "lr": float(best["lr"]),
    "weight_decay": float(best["weight_decay"]),
    "max_epochs": int(best["max_epochs"]),
    "patience": int(best["patience"]),
}

early_stopping_metric = safe_get_str(best, "early_stopping_metric", "f1")
if early_stopping_metric not in ["loss", "auroc", "f1"]:
    print(f"[Warning] Invalid early_stopping_metric={early_stopping_metric}; fallback to 'f1'")
    early_stopping_metric = "f1"

print("\nRetraining TabM with cfg:")
print(cfg)
print("early_stopping_metric:", early_stopping_metric)

set_seed(RANDOM_SEED)

result = train_one_tabm_setting_for_bootstrap(
    cfg,
    early_stopping_metric=early_stopping_metric,
)

valid_labels = result["valid_labels"]
valid_probs = result["valid_probs"]
test_labels = result["test_labels"]
test_probs = result["test_probs"]

valid_f1_thresh, valid_best_f1 = find_best_f1_threshold(
    valid_labels,
    valid_probs,
    grid_size=101,
)

if THRESHOLD_MODE_FOR_BOOTSTRAP == "valid_f1":
    threshold = valid_f1_thresh
elif THRESHOLD_MODE_FOR_BOOTSTRAP == "csv":

    if "valid_threshold" in best and not pd.isna(best["valid_threshold"]):
        threshold = float(best["valid_threshold"])
    elif "valid_f1_threshold" in best and not pd.isna(best["valid_f1_threshold"]):
        threshold = float(best["valid_f1_threshold"])
    else:
        threshold = 0.5
elif THRESHOLD_MODE_FOR_BOOTSTRAP == "0.5":
    threshold = 0.5
else:
    raise ValueError("THRESHOLD_MODE_FOR_BOOTSTRAP must be 'valid_f1', 'csv', or '0.5'.")

threshold_info = {
    "threshold_mode_for_bootstrap": THRESHOLD_MODE_FOR_BOOTSTRAP,
    "threshold": float(threshold),
    "valid_f1_thresh_recomputed": float(valid_f1_thresh),
    "valid_best_f1_at_threshold_recomputed": float(valid_best_f1),
    "csv_valid_threshold": safe_get_float(best, "valid_threshold", 0.5),
    "csv_valid_f1_threshold": safe_get_float(best, "valid_f1_threshold", 0.5),
    "best_epoch_retrained": int(result["best_epoch"]),
    "best_monitor_retrained": float(result["best_monitor"]),
}

print("\nThreshold info:")
print(json.dumps(threshold_info, indent=2))


# ============================================================
# 10. Save predictions
# ============================================================

pred_df = pd.DataFrame({
    "CaseNo": test_ids,
    "true_label": test_labels.astype(int),
    "predicted_prob": test_probs.astype(float),
    "predicted_class_0.5": (test_probs >= 0.5).astype(int),
    "predicted_class_valid_f1": (test_probs >= valid_f1_thresh).astype(int),
    "predicted_class_bootstrap_threshold": (test_probs >= threshold).astype(int),
})

pred_path = os.path.join(OUTPUT_DIR, "tabm_predictions_test_retrained_best_valid.csv")
pred_df.to_csv(pred_path, index=False)

with open(os.path.join(OUTPUT_DIR, "tabm_thresholds_retrained_best_valid.json"), "w") as f:
    json.dump(threshold_info, f, indent=2)

point_metrics = calc_metrics(test_labels, test_probs, threshold)

print("\nPoint estimate on test set:")
for k, v in point_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")


# ============================================================
# 11. Bootstrap test predictions
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

indices = np.arange(len(test_labels))
pos_idx = indices[test_labels == 1]
neg_idx = indices[test_labels == 0]

rows = []

for b in range(N_BOOTSTRAP):
    if STRATIFIED:
        sampled_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        sampled_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        sampled_idx = np.concatenate([sampled_pos, sampled_neg])
        rng.shuffle(sampled_idx)
    else:
        sampled_idx = rng.choice(indices, size=len(indices), replace=True)

    y_b = test_labels[sampled_idx]
    p_b = test_probs[sampled_idx]

    m = calc_metrics(y_b, p_b, threshold)
    m["bootstrap_id"] = b
    rows.append(m)

boot_df = pd.DataFrame(rows)

boot_path = os.path.join(OUTPUT_DIR, "tabm_bootstrap_test_samples.csv")
boot_df.to_csv(boot_path, index=False)


# ============================================================
# 12. Bootstrap 95% CI summary
# ============================================================

metric_cols = [
    "auroc",
    "auprc",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "positive_rate",
]

summary_rows = []

for m in metric_cols:
    values = boot_df[m].replace([np.inf, -np.inf], np.nan).dropna().values

    summary_rows.append({
        "metric": m,
        "point": point_metrics[m],
        "boot_mean": float(np.mean(values)),
        "boot_std": float(np.std(values, ddof=1)),
        "ci_lower_2.5": float(np.percentile(values, 2.5)),
        "ci_upper_97.5": float(np.percentile(values, 97.5)),
        "n_bootstrap_valid": int(len(values)),
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(OUTPUT_DIR, "tabm_bootstrap_test_summary.csv")
summary_df.to_csv(summary_path, index=False)

summary_json = {
    "selected_by": BEST_SETTING_SORT,
    "threshold_mode_for_bootstrap": THRESHOLD_MODE_FOR_BOOTSTRAP,
    "threshold": float(threshold),
    "n_bootstrap": int(N_BOOTSTRAP),
    "stratified": bool(STRATIFIED),
    "selected_setting": {k: str(v) for k, v in best.items()},
    "retrained_cfg": cfg,
    "early_stopping_metric": early_stopping_metric,
    "threshold_info": threshold_info,
    "point_metrics": point_metrics,
    "summary": summary_df.to_dict(orient="records"),
}

summary_json_path = os.path.join(OUTPUT_DIR, "tabm_bootstrap_test_summary.json")

with open(summary_json_path, "w") as f:
    json.dump(make_jsonable(summary_json), f, indent=2)

print("\nSaved:")
print(pred_path)
print(boot_path)
print(summary_path)
print(summary_json_path)

print("\nBootstrap 95% CI summary:")
display(summary_df)

# cleanup
try:
    del result
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
except Exception:
    pass

TabPFN

In [ ]:
# Install sklearn
!pip install scikit-learn==1.5.2

# Install TabPFN
!pip install tabpfn
!pip install tabpfn-client
# TabPFN Community installs optional functionalities around the TabPFN model
# These include post-hoc ensembles, interpretability tools, and more
!git clone https://github.com/PriorLabs/tabpfn-extensions
!pip install -e tabpfn-extensions[all]

In [ ]:
import os
import pandas as pd
from tabpfn import TabPFNRegressor
from sklearn.metrics import mean_squared_error, r2_score

def prepare_features(df, target_col, is_train=True, encoder_dict=None):
    """Automatically encode categorical features safely."""
    df_copy = df.copy()
    X = df_copy.drop(columns=[target_col])
    y = df_copy[target_col]

    if encoder_dict is None:
        encoder_dict = {}

    categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns

    for col in categorical_cols:
        if is_train:
            labels, uniques = pd.factorize(X[col])
            X[col] = labels
            encoder_dict[col] = uniques
        else:
            if col in encoder_dict:
                uniques = encoder_dict[col]
                mapping = {val: i for i, val in enumerate(uniques)}
                X[col] = X[col].map(mapping).fillna(-1).astype(int)
            else:
                X[col] = pd.factorize(X[col])[0]

    return X, y, encoder_dict

def main():
    # Setup data directories
    data_dir = "<path of data>"
    train_path = os.path.join(data_dir, "ml_train_both_v3.csv")
    valid_path = os.path.join(data_dir, "ml_valid_both_v3.csv")
    test_path = os.path.join(data_dir, "ml_test_both_v3.csv")

    print("Reading v3 datasets...")
    train_df = pd.read_csv(train_path)
    valid_df = pd.read_csv(valid_path)
    test_df = pd.read_csv(test_path)

    # Precisely lock target column alignment
    target_col = "EWAS_label"
    print(f"Target column successfully locked: {target_col}")

    # Feature engineering for categorical data
    print("Processing feature profiles and encoding categorical columns...")
    X_train, y_train, encoders = prepare_features(train_df, target_col, is_train=True)
    X_valid, y_valid, _ = prepare_features(valid_df, target_col, is_train=False, encoder_dict=encoders)
    X_test, y_test, _ = prepare_features(test_df, target_col, is_train=False, encoder_dict=encoders)

    # Initialize TabPFN-3 Regressor
    model_v3_name = "tabpfn-v3-regressor-v3_default.ckpt"
    print(f"Loading latest TabPFN-3 regressor configuration: {model_v3_name}...")

    classifier = TabPFNRegressor(
        device='auto',
        model_path=model_v3_name
    )

    # Model fitting
    print("Fitting model via TabPFN-3 mixed-feature in-context learning...")
    classifier.fit(X_train, y_train)

    # Validation evaluation
    print("Generating validation set inferences...")
    valid_preds = classifier.predict(X_valid)
    print(f"✅ Validation Mean Squared Error (MSE): {mean_squared_error(y_valid, valid_preds):.4f}")
    print(f"✅ Validation R² Coefficient of Determination: {r2_score(y_valid, valid_preds):.4f}")

    # Test evaluation
    print("Generating test set inferences...")
    test_preds = classifier.predict(X_test)
    test_mse = mean_squared_error(y_test, test_preds)
    test_r2 = r2_score(y_test, test_preds)

    print(f"\n✅ Test Mean Squared Error (MSE): {test_mse:.4f}")
    print(f"✅ Test R² Coefficient of Determination: {test_r2:.4f}")

# ==========================================
# System Integrity Verification Module
# ==========================================
def test_pipeline_readiness():
    """
    System check framework: Verifies environmental dependencies and structural integrity
    of the ingestion pipeline before executing model allocation.
    """
    print("\n[🛠️ SYSTEM CHECK] Initiating pre-execution environment verification...")

    # 1. Verify environmental token allocation
    token = os.environ.get("TABPFN_TOKEN", "")
    if not token or token.startswith("your_"):
        print("[ERROR] Verification Failed: No valid TABPFN_TOKEN detected in environment!")
        return False

    # 2. Structural column and integrity check
    data_dir = "mnt/data"
    target_files = ["ml_train_both_v3.csv", "ml_valid_both_v3.csv", "ml_test_both_v3.csv"]
    target_col = "EWAS_label"

    for f in target_files:
        f_path = os.path.join(data_dir, f)
        # Check physical existence
        if not os.path.exists(f_path):
            print(f"[ERROR] Verification Failed: File path target missing -> {f_path}")
            return False

        # Structural testing via chunk ingestion to safeguard against corrupted/missing layouts
        try:
            df_check = pd.read_csv(f_path, nrows=5)
            if target_col not in df_check.columns:
                print(f"[ERROR] Structural Exception: Specified target '{target_col}' missing in '{f}'! Found trailing column: '{df_check.columns[-1]}'")
                return False
        except Exception as e:
            print(f"[ERROR] Ingestion Test Failure on dataset {f}: {e}")
            return False

    print("[SUCCESS] Pipeline verification and environmental integrity assertions passed. Proceeding to execution.")
    return True

if __name__ == "__main__":
    # 🌟 FIX: Token injection must execute BEFORE running integrity checks
    os.environ["TABPFN_TOKEN"] = "<TABPFN TOKEN>"

    if test_pipeline_readiness():
        main()

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from tabpfn import TabPFNClassifier
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample

# ==========================================
# Core Feature Engineering Module
# ==========================================
def prepare_features(df, target_col="EWAS_label", is_train=True, encoder_dict=None):
    df_copy = df.copy()

    if target_col and target_col in df_copy.columns:
        X = df_copy.drop(columns=[target_col])
        y = df_copy[target_col]
    else:
        X = df_copy
        y = None

    if encoder_dict is None:
        encoder_dict = {}

    categorical_cols = X.select_dtypes(exclude=['number']).columns

    for col in categorical_cols:
        X[col] = X[col].astype(str)
        if is_train:
            labels, uniques = pd.factorize(X[col])
            X[col] = labels
            encoder_dict[col] = uniques
        else:
            if col in encoder_dict:
                uniques = encoder_dict[col]
                mapping = {val: i for i, val in enumerate(uniques)}
                X[col] = X[col].map(mapping).fillna(-1).astype(int)
            else:
                X[col] = pd.factorize(X[col])[0]

    X = X.astype(float)
    return X, y, encoder_dict

# ==========================================
# Validation Threshold Optimization Module
# ==========================================
def find_best_f1_threshold(y_true, y_prob, num_thresholds=100):
    """
    Finds the optimal classification threshold that maximizes the F1-score on the
    validation set to strictly prevent data leakage.
    """
    best_thresh = 0.5
    best_f1 = 0.0
    # Generate candidate threshold values between 0.01 and 0.99
    thresholds = np.linspace(0.01, 0.99, num_thresholds)

    for thresh in thresholds:
        preds = (y_prob >= thresh).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh

    return best_thresh

# ==========================================
# Bootstrap Metrics Evaluation Module
# ==========================================
def calculate_bootstrap_metrics(y_true, y_prob, dataset_name, threshold, n_bootstraps=1000):
    """
    Computes evaluation metrics (Mean ± Std) using bootstrapping to match the
    reporting format required for the manuscript.
    """
    print(f"Initiating bootstrapping for the {dataset_name} set ({n_bootstraps} iterations, applying fixed threshold: {threshold:.4f})...")

    # Tracking target metrics aligning with the paper: AUROC, AUPRC, F1, and Recall
    metrics = {'AUROC': [], 'AUPRC': [], 'F1': [], 'Recall': []}

    y_true_np = np.array(y_true)
    y_prob_np = np.array(y_prob)

    rng = np.random.RandomState(42)

    for i in range(n_bootstraps):
        # Random resampling with replacement
        indices = resample(np.arange(len(y_true_np)), random_state=rng.randint(0, 100000))

        y_true_b = y_true_np[indices]
        y_prob_b = y_prob_np[indices]

        # Ensure bootstrap sample contains at least two classes for metric computation
        if len(np.unique(y_true_b)) < 2:
            continue

        # Map predictions using the pre-determined validation threshold
        y_pred_b = (y_prob_b >= threshold).astype(int)

        metrics['AUROC'].append(roc_auc_score(y_true_b, y_prob_b))
        metrics['AUPRC'].append(average_precision_score(y_true_b, y_prob_b))
        metrics['F1'].append(f1_score(y_true_b, y_pred_b, zero_division=0))
        metrics['Recall'].append(recall_score(y_true_b, y_pred_b, zero_division=0))

    results = {'Dataset': dataset_name, 'Selected_Threshold': f"{threshold:.4f}"}
    for metric_name, values in metrics.items():
        if values:
            mean_val = np.mean(values)
            std_val = np.std(values)  # Format directly as Mean ± Std for LaTeX compatibility
            results[metric_name] = f"{mean_val:.3f} ± {std_val:.3f}"
        else:
            results[metric_name] = "N/A"

    return results

# ==========================================
# Isolated Unit Verification Module
# ==========================================
def test_pipeline():
    """
    System check framework: Verifies the integrity of threshold locks and Recall calculation
    mechanisms using mock configurations before runtime allocation.
    """
    print("\n[🛠️ Executing Unit Test] Verifying threshold locking and Recall calculation logic...")

    # Construct 100 entries of mock imbalanced binary data
    np.random.seed(42)
    mock_y_val = np.random.choice([0, 1], size=100, p=[0.8, 0.2])
    mock_prob_val = np.random.uniform(0.0, 0.8, size=100)

    mock_y_test = np.random.choice([0, 1], size=100, p=[0.8, 0.2])
    mock_prob_test = np.random.uniform(0.0, 0.8, size=100)

    # Test 1: Verify threshold boundary localization
    best_thresh = find_best_f1_threshold(mock_y_val, mock_prob_val)
    assert 0.0 <= best_thresh <= 1.0, "Assertion Error: Optimized threshold falls outside valid bounds!"

    # Test 2: Validate bootstrap stability and string mapping format
    test_res = calculate_bootstrap_metrics(mock_y_test, mock_prob_test, "MockTest", best_thresh, n_bootstraps=10)

    assert 'Recall' in test_res, "Assertion Error: Recall metric calculation missing from output keys!"
    assert '±' in test_res['Recall'], "Assertion Error: Computed layout does not follow 'Mean ± Std' schema!"

    print("✅ [Unit Test Passed] Core metrics evaluation and anti-leakage systems are fully sound.")


# ==========================================
# Execution Entry Point
# ==========================================
if __name__ == "__main__":
    # Execute structural system checks before runtime setup
    test_pipeline()

    # 🔴 Insert API Token
    os.environ["TABPFN_TOKEN"] = "<TABPFN TOKEN>"

    TARGET_COLUMN_NAME = "EWAS_label"
    data_dir = "<path of data>"
    output_dir = "<output path>"
    os.makedirs(output_dir, exist_ok=True)

    train_path = os.path.join(data_dir, "ml_train_both_v3.csv")
    valid_path = os.path.join(data_dir, "ml_valid_both_v3.csv")
    test_path = os.path.join(data_dir, "ml_test_both_v3.csv")

    csv_report_path = os.path.join(output_dir, "bootstrap_metrics_report.csv")
    inference_output_path = os.path.join(output_dir, "tabpfm_test_predictions.csv")

    try:
        # 1. Data Loading & Cleaning
        print("\n--- [Phase 1] Data Ingestion and Preprocessing ---")
        train_df = pd.read_csv(train_path).dropna(subset=[TARGET_COLUMN_NAME])
        valid_df = pd.read_csv(valid_path).dropna(subset=[TARGET_COLUMN_NAME])
        test_df = pd.read_csv(test_path).dropna(subset=[TARGET_COLUMN_NAME])

        # 2. Feature & Label Pipeline
        X_train, y_train_raw, encoders = prepare_features(train_df, TARGET_COLUMN_NAME, is_train=True)
        X_valid, y_valid_raw, _ = prepare_features(valid_df, TARGET_COLUMN_NAME, is_train=False, encoder_dict=encoders)
        X_test, y_test_raw, _ = prepare_features(test_df, TARGET_COLUMN_NAME, is_train=False, encoder_dict=encoders)

        label_encoder = LabelEncoder()
        y_train = label_encoder.fit_transform(y_train_raw.astype(str))
        y_valid = label_encoder.transform(y_valid_raw.astype(str))
        y_test = label_encoder.transform(y_test_raw.astype(str))

        # 3. Model Training
        print("\n--- [Phase 2] Initializing TabPFN Fit ---")
        model = TabPFNClassifier(device='auto')
        model.fit(X_train, y_train)

        # 4. Probabilistic Inference Setup (Transitioned to probability space to bypass default 0.5 cuts)
        print("\n--- [Phase 3] Generating Prediction Probabilities ---")
        valid_probs = model.predict_proba(X_valid)[:, 1]
        test_probs = model.predict_proba(X_test)[:, 1]

        # Anti-leakage checkpoint: Isolate operational boundary optimization exclusively within validation space
        print("\nLocking optimal operational threshold...")
        best_validation_threshold = find_best_f1_threshold(y_valid, valid_probs)
        print(f"🎯 Validation optimal F1 threshold locked at: {best_validation_threshold:.4f}")

        # Deploy finalized threshold lock to extract Test predictions
        test_final_preds = (test_probs >= best_validation_threshold).astype(int)
        test_df['Predicted_Label'] = label_encoder.inverse_transform(test_final_preds)
        test_df['Confidence_Prob'] = test_probs.round(4)
        test_df.to_csv(inference_output_path, index=False)
        print(f"✅ Detailed test set inference successfully saved to: {inference_output_path}")

        # 5. Bootstrapping Strategy & Compilation
        print("\n--- [Phase 4] Executing Bootstrap Metric Distribution ---")
        valid_metrics = calculate_bootstrap_metrics(y_valid, valid_probs, dataset_name="Validation", threshold=best_validation_threshold)
        test_metrics = calculate_bootstrap_metrics(y_test, test_probs, dataset_name="Test", threshold=best_validation_threshold)

        # Merge results into output structure
        report_df = pd.DataFrame([valid_metrics, test_metrics])

        # Display performance logs
        print("\n📊 Bootstrapping Evaluation Summary (Mean ± Std):")
        print(report_df.to_string(index=False))

        # Export serialization metrics
        report_df.to_csv(csv_report_path, index=False)
        print(f"\n🚀 Execution pipeline complete. Metrics matrix logged at: {csv_report_path}")

    except Exception as e:
        print(f"\n[❌ Pipeline Failure] Error log trace: {e}")